        # 03 Gale–Shapley 法と安定マッチング

        - **Deferred Acceptance**（受け入れ保留方式）の動きを、提案・仮マッチング・拒否・再提案のステップで見る
        - **Blocking pair**（互いに今の相手より好む 2 人）を数え、数学的な**安定性**を確かめる（RQ6 の前半）

> **教育用の架空データ・架空モデル**です。参加者は実在の人物ではなく、Group A・B はアルゴリズム上の区分で性別や性的指向を意味しません。希望スコアは実際の結婚適性を表さず、関係継続モデルの係数は実際の離婚率や実証された因果効果ではありません。

## 準備：実験環境（最初に必ず実行）

F3 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F3 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F3 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F3 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                              # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f3-matching-algorithms"     # リポジトリの中の F3 フォルダ（ルートが F3 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f3_root(p):
    return (Path(p) / "matching_algorithms.py").exists() and (Path(p) / "relationship_stability.py").exists()


def find_f3_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F3 のルート
        if is_f3_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f3_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F3 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f3_upload")
    for hit in glob.glob("/content/f3_upload/**/matching_algorithms.py", recursive=True):
        if is_f3_root(Path(hit).parent):
            return Path(hit).parent
    return None


F3_ROOT = find_f3_root()
if F3_ROOT is None:
    raise RuntimeError("F3 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F3_ROOT)
if str(F3_ROOT) not in sys.path:
    sys.path.insert(0, str(F3_ROOT))
if not (F3_ROOT / "textbook" / "fixed_queue.py").exists():
    print("⚠️ textbook/ がありません。preferences.py・matching_algorithms.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import participants as pt
import preferences as pr
import matching_algorithms as ma
import relationship_stability as rs
import evaluation as ev
import matching_charts as mc

mc.setup_style()                       # 文字を大きくするなら mc.setup_style(1.3)
groups = pt.load_participants()
scores = pr.score_table(groups)
rankings = pr.build_rankings(groups, scores)
params = rs.load_parameters()          # 関係継続モデルのパラメータ（すべて架空）
A_IDS = [p["id"] for p in groups["A"]]
B_IDS = [p["id"] for p in groups["B"]]
N_TOTAL = pt.total_participants(groups)
compat = lambda a, b: pr.pair_compatibility(a, b, scores)

SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", mc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        heads = [h for _k, h, _f in columns]
        widths = [max(len(h), *(len(d[h]) for d in data)) for h in heads]
        print("  ".join(h.rjust(w) for h, w in zip(heads, widths)))
        for d in data:
            print("  ".join(d[h].rjust(w) for h, w in zip(heads, widths)))


print("F3_ROOT:", F3_ROOT)
print(f"参加者 Group A {len(A_IDS)} 人・Group B {len(B_IDS)} 人 / 関係継続モデル（架空）:",
      {k: params[k] for k in ("baseline_hazard", "alpha", "beta", "gamma", "default_annual_stress",
                               "default_conflict_resolution", "simulation_years", "random_seed")})

## アルゴリズム（Group A が提案する）

1. 相手のいない提案者を**キュー**に入れる（教科書 第4章 `FixedQueue`）
2. キューから 1 人取り出し、まだ提案していない相手のうち**希望順位が最も高い人**に提案する（受け入れ可能な相手だけ、同じ相手には 1 回まで）
3. 受け手は、仮の相手がいなければ**仮に受け入れる**。いれば、**順位辞書**（ハッシュ表）で比べて好む方を残し、もう一方を**拒否**する
4. 拒否された人は、まだ提案できる相手がいれば**キューに戻り再提案**する
5. キューが空になったら終了。仮マッチングが**最終マッチング**になる

## 希望順位

In [ ]:
for pid, lst in rankings.items():
    print(f"{pid}: {' > '.join(lst) or '(nobody)'}")

## ステップごとの変化

In [ ]:
gs, steps = ma.gale_shapley(rankings, A_IDS, B_IDS, record=True)
table(steps, [("step", "step", "{}"), ("proposer", "proposer", "{}"), ("receiver", "receiver", "{}"),
              ("action", "action", "{}"), ("rejected", "rejected", "{}"), ("tentative", "tentative matches", "{}")])

In [ ]:
#@title グラフを表示（コードは隠しています）
show(mc.plot_gs_steps(steps, A_IDS, B_IDS), "03_gs_steps")

## 最終マッチングと Blocking pair

Gale–Shapley と Greedy の結果を並べ、Blocking pair（赤の破線）を比べます。

In [ ]:
greedy = ma.greedy_matching(rankings, A_IDS, B_IDS)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.8))
for ax, (name, m) in zip(axes, [("Greedy", greedy), ("Gale-Shapley", gs)]):
    bp = ev.blocking_pairs(m, rankings, A_IDS, B_IDS)
    mc.draw_bipartite(ax, A_IDS, B_IDS, m, f"{name}: {len(bp)} blocking pair(s)", blocking=bp, compat=compat)
    print(f"{name:13}: {dict(sorted(m.items()))}  blocking pairs = {bp}")
fig.tight_layout()
show(fig, "03_final_blocking")

## 数学的な Stable Matching と、長期的な Relationship Stability は違う

| | 数学的な安定性（Stable Matching） | 関係の継続（Relationship Stability） |
| --- | --- | --- |
| 意味 | マッチングの**時点**で、互いに今の相手より好む 2 人（Blocking pair）が**いない** | 成立した関係が**時間とともに**続くか |
| 何で決まるか | 参加者の**希望順位**だけ | この教材では、初期適合度・葛藤対処・ストレスと**乱数**（架空モデル、Notebook 04） |
| 確かめ方 | Blocking pair を数える（O(N²)） | 時系列シミュレーション（O(PT)、反復 M 回） |

- Gale–Shapley 法が保証するのは**左側だけ**です。Blocking pair が 0 でも、**関係が長続きすることは保証しません**。
- 「安定（stable）」という同じ言葉を使っていても、**まったく別の概念**です。報告では区別して書いてください（RQ6）。
- 2 つの関係は、Notebook 05 で**シミュレーションの結果**から調べます。

## 学生向け TODO・考察

- [ ] Group B が提案する Gale–Shapley（`ma.swap_sides`）と比べ、どちらのグループの満足度が高いか調べる
- [ ] 希望順位を変えて、Greedy に Blocking pair ができる条件・できない条件を探す
- [ ] 提案の回数を数え、参加者数を増やしたときの増え方（O(N²)）を確かめる

**考察（チームで記入）**：TODO

## 参考文献

- D. Gale, L. S. Shapley, "College Admissions and the Stability of Marriage", *The American Mathematical Monthly*, 69(1), 9–15, 1962.
- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 線形探索・ハッシュ法、第4章 キュー、第6章 挿入ソート）
- F3 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_participants.json`、`data/relationship_parameters.json`（この教材のための**架空のデータ**）